# 1-2. RAG Indexing용 PDF → Markdown 전처리

## 학습 목표

- PDF를 Markdown으로 변환한다.
- `text_cleaner.py`를 import해서 페이지 단위 전처리를 수행한다.
- 공백 라인만 제거하고 실제 줄 구조는 유지한다.
- Markdown table 구조를 손상하지 않는다.
- 페이지 단위 `Document`를 생성해 RAG indexing에 사용한다.

## 기준

- 데이터 경로: `../data`
- 출력 경로: `../data`
- 전처리 모듈: `./text_cleaner.py`
- 청크 단위: `1 page = 1 chunk`

## 0. 필요 라이브러리 설치

In [ ]:
# uv 환경
# !uv add pymupdf4llm langchain-core pandas

# pip 환경
# %pip install pymupdf4llm langchain-core pandas

## 1. Import 및 경로 설정

In [ ]:
from pathlib import Path
import importlib

import pymupdf4llm
import pandas as pd
from langchain_core.documents import Document

import text_cleaner
importlib.reload(text_cleaner)

PDF_NAME = "공직자_민원응대_핵심_매뉴얼.pdf"

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../data")

PDF_PATH = DATA_DIR / PDF_NAME

RAW_MD_PATH = OUTPUT_DIR / f"{PDF_PATH.stem}_raw.md"
CLEAN_MD_PATH = OUTPUT_DIR / f"{PDF_PATH.stem}_cleaned.md"
CHUNK_MD_PATH = OUTPUT_DIR / f"{PDF_PATH.stem}_rag_page_chunks.md"
CHUNK_CSV_PATH = OUTPUT_DIR / f"{PDF_PATH.stem}_rag_page_chunks.csv"
QUALITY_CSV_PATH = OUTPUT_DIR / f"{PDF_PATH.stem}_preprocess_quality.csv"

if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF 파일을 찾을 수 없습니다: {PDF_PATH}")

print("PDF_PATH:", PDF_PATH)
print("text_cleaner:", text_cleaner.__file__)
print("CLEAN_MD_PATH:", CLEAN_MD_PATH)

## 2. 메타데이터 및 점검 함수 정의

In [ ]:
def infer_section(page: int) -> str:
    """페이지 번호로 문서 내 섹션을 추론한다."""
    if page <= 4:
        return "민원응대 관련 기본원칙"
    if page <= 7:
        return "일반적인 민원 응대요령"
    if page <= 12:
        return "특이민원 응대요령"
    if page <= 14:
        return "민원담당자 회복 및 보호조치"
    if page <= 17:
        return "질의응답"
    return "참고자료"


def infer_topic(text: str) -> str:
    """키워드 기반으로 페이지의 대표 주제를 추론한다."""
    rules = [
        ("반복전화", ["반복 전화", "반복전화", "동일민원"]),
        ("장시간 통화", ["장시간", "권장시간", "20분", "15분"]),
        ("상급자 통화 요구", ["상급자", "기관장", "부서장"]),
        ("폭언", ["폭언", "욕설", "협박", "성희롱"]),
        ("폭행", ["폭행", "공무집행방해"]),
        ("물품 파손", ["파손", "집기", "공용물건"]),
        ("위험물 소지", ["위험물", "흉기", "119"]),
        ("보호조치", ["휴식", "휴가", "심리상담", "치료"]),
        ("법적 근거", ["형법", "경범죄", "스토킹", "적용법률"]),
    ]

    for topic, keywords in rules:
        if any(keyword in text for keyword in keywords):
            return topic

    return "일반 민원응대"


def extract_markdown_text(item) -> str:
    """pymupdf4llm의 page_chunks 반환값에서 Markdown 텍스트를 추출한다."""
    if isinstance(item, str):
        return item

    if isinstance(item, dict):
        return item.get("text") or item.get("markdown") or item.get("content") or ""

    return str(item)


def make_markdown_document(pages: list[str], title: str) -> str:
    """페이지별 Markdown을 하나의 Markdown 문서로 결합한다."""
    lines = [f"# {title}"]

    for page_no, text in enumerate(pages, start=1):
        lines.append(f"<!-- page: {page_no} -->")
        lines.extend(text.strip().splitlines())

    # 공백 라인만 제거한다.
    return "\\n".join(line.rstrip() for line in lines if line.strip()) + "\\n"


def count_blank_lines(text: str) -> int:
    """공백 라인 수를 계산한다."""
    return sum(1 for line in text.splitlines() if not line.strip())


def count_table_lines(text: str) -> int:
    """Markdown table row 수를 계산한다."""
    return sum(1 for line in text.splitlines() if line.strip().count("|") >= 2)


def count_heading_lines(text: str) -> int:
    """Markdown heading 수를 계산한다."""
    return sum(1 for line in text.splitlines() if line.strip().startswith("#"))


def count_dot_sequences(text: str) -> int:
    """점선성 문자 패턴 수를 계산한다."""
    import re
    return len(re.findall(r"\\.{3,}|…+", text))


def make_quality_record(page: int, raw: str, cleaned: str) -> dict:
    """페이지별 전처리 품질 점검 record를 만든다."""
    return {
        "page": page,
        "raw_chars": len(raw),
        "cleaned_chars": len(cleaned),
        "raw_lines": len(raw.splitlines()),
        "cleaned_lines": len(cleaned.splitlines()),
        "raw_blank_lines": count_blank_lines(raw),
        "cleaned_blank_lines": count_blank_lines(cleaned),
        "raw_dot_sequences": count_dot_sequences(raw),
        "cleaned_dot_sequences": count_dot_sequences(cleaned),
        "raw_table_lines": count_table_lines(raw),
        "cleaned_table_lines": count_table_lines(cleaned),
        "raw_heading_lines": count_heading_lines(raw),
        "cleaned_heading_lines": count_heading_lines(cleaned),
    }

## 3. PDF를 페이지별 Markdown으로 변환

In [ ]:
md_result = pymupdf4llm.to_markdown(
    str(PDF_PATH),
    page_chunks=True
)

if isinstance(md_result, str):
    raw_page_texts = [md_result]
else:
    raw_page_texts = [extract_markdown_text(page) for page in md_result]

raw_page_texts = [text for text in raw_page_texts if text and text.strip()]

raw_markdown = make_markdown_document(
    raw_page_texts,
    title=f"{PDF_PATH.stem} Raw Markdown"
)

RAW_MD_PATH.write_text(raw_markdown, encoding="utf-8")

print("페이지 수:", len(raw_page_texts))
print("raw Markdown 저장:", RAW_MD_PATH)

## 4. Markdown 구조 보존 전처리

In [ ]:
cleaned_page_texts = text_cleaner.clean_pages(raw_page_texts)

cleaned_markdown = make_markdown_document(
    cleaned_page_texts,
    title=f"{PDF_PATH.stem} Cleaned Markdown"
)

CLEAN_MD_PATH.write_text(cleaned_markdown, encoding="utf-8")

print("cleaned Markdown 저장:", CLEAN_MD_PATH)
print("cleaned 페이지 수:", len(cleaned_page_texts))
print("cleaned Markdown 공백 라인 수:", count_blank_lines(cleaned_markdown))

## 5. 페이지 단위 Document 생성

In [ ]:
page_docs = []

for page_no, text in enumerate(cleaned_page_texts, start=1):
    page_docs.append(
        Document(
            page_content=text,
            metadata={
                "source": PDF_PATH.name,
                "page": page_no,
                "section": infer_section(page_no),
                "topic": infer_topic(text),
                "chunk_type": "page",
                "table_line_count": count_table_lines(text),
                "heading_count": count_heading_lines(text),
                "blank_line_count": count_blank_lines(text),
                "char_count": len(text),
            }
        )
    )

chunks = page_docs

print("page_docs 수:", len(page_docs))
print("chunks 수:", len(chunks))
print(chunks[0].metadata)
print(chunks[0].page_content[:1000])

## 6. 전처리 품질 점검

In [ ]:
quality_records = [
    make_quality_record(page_no, raw, cleaned)
    for page_no, (raw, cleaned) in enumerate(zip(raw_page_texts, cleaned_page_texts), start=1)
]

quality_df = pd.DataFrame(quality_records)
quality_df.to_csv(QUALITY_CSV_PATH, index=False, encoding="utf-8-sig")

quality_df[[
    "page",
    "raw_chars",
    "cleaned_chars",
    "raw_blank_lines",
    "cleaned_blank_lines",
    "raw_dot_sequences",
    "cleaned_dot_sequences",
    "raw_table_lines",
    "cleaned_table_lines",
    "raw_heading_lines",
    "cleaned_heading_lines",
]]

## 7. RAG page chunk 저장

In [ ]:
chunk_lines = [f"# {PDF_PATH.stem} RAG Page Chunks"]

for doc in chunks:
    meta = doc.metadata

    chunk_lines.append(f"## Page {meta['page']} | {meta['section']} | {meta['topic']}")
    chunk_lines.append(f"<!-- source: {meta['source']} -->")
    chunk_lines.append(f"<!-- chunk_type: {meta['chunk_type']} -->")
    chunk_lines.append(f"<!-- table_line_count: {meta['table_line_count']} -->")
    chunk_lines.append(f"<!-- blank_line_count: {meta['blank_line_count']} -->")
    chunk_lines.append(f"<!-- char_count: {meta['char_count']} -->")
    chunk_lines.extend(doc.page_content.splitlines())

chunk_markdown = "\n".join(line.rstrip() for line in chunk_lines if line.strip()) + "\n"
CHUNK_MD_PATH.write_text(chunk_markdown, encoding="utf-8")

chunk_df = pd.DataFrame([
    {
        "page": doc.metadata["page"],
        "section": doc.metadata["section"],
        "topic": doc.metadata["topic"],
        "chunk_type": doc.metadata["chunk_type"],
        "table_line_count": doc.metadata["table_line_count"],
        "heading_count": doc.metadata["heading_count"],
        "blank_line_count": doc.metadata["blank_line_count"],
        "char_count": doc.metadata["char_count"],
        "content": doc.page_content,
    }
    for doc in chunks
])

chunk_df.to_csv(CHUNK_CSV_PATH, index=False, encoding="utf-8-sig")

print("chunk Markdown 저장:", CHUNK_MD_PATH)
print("chunk CSV 저장:", CHUNK_CSV_PATH)
print("chunk Markdown 공백 라인 수:", count_blank_lines(chunk_markdown))

## 8. 표 포함 chunk 확인

In [ ]:
table_chunks = [
    doc for doc in chunks
    if doc.metadata["table_line_count"] > 0
]

print("표 포함 chunk 수:", len(table_chunks))

if table_chunks:
    sample = table_chunks[0]
    print(sample.metadata)
    print(sample.page_content[:2000])

## 9. chunk 통계 확인

In [ ]:
chunk_df[[
    "page",
    "section",
    "topic",
    "table_line_count",
    "heading_count",
    "blank_line_count",
    "char_count",
]]